In [8]:
# ============================================
# 1. LOAD ML DATA
# ============================================

import pandas as pd
import numpy as np
import joblib

# Load train/test data
X_train = joblib.load("../data/X_train.pkl")
X_test = joblib.load("../data/X_test.pkl")

y_train = joblib.load("../data/y_train.pkl")
y_test = joblib.load("../data/y_test.pkl")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (144415, 273)
X_test: (36104, 273)
y_train: (144415,)
y_test: (36104,)


In [11]:
# ============================================
# 2. TRAIN LEAKAGE-SAFE LOGISTIC REGRESSION
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

logistic_model = LogisticRegression(
    max_iter=1500,
    random_state=42
)

print("Training Logistic Regression...")

logistic_model.fit(X_train, y_train)

# Predictions
y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Evaluation
log_accuracy = accuracy_score(y_test, y_pred)
log_precision = precision_score(y_test, y_pred)
log_recall = recall_score(y_test, y_pred)
log_f1 = f1_score(y_test, y_pred)
log_roc_auc = roc_auc_score(y_test, y_prob)

print("\nLogistic Regression Performance")
print("--------------------------------")
print(f"Accuracy : {log_accuracy:.4f}")
print(f"Precision: {log_precision:.4f}")
print(f"Recall   : {log_recall:.4f}")
print(f"F1 Score : {log_f1:.4f}")
print(f"ROC-AUC  : {log_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training Logistic Regression...

Logistic Regression Performance
--------------------------------
Accuracy : 0.6965
Precision: 0.8481
Recall   : 0.5439
F1 Score : 0.6628
ROC-AUC  : 0.7426

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.88      0.72     16308
           1       0.85      0.54      0.66     19796

    accuracy                           0.70     36104
   macro avg       0.73      0.71      0.69     36104
weighted avg       0.74      0.70      0.69     36104



c:\Users\mjana\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
# ============================================
# 3. TRAIN LEAKAGE-SAFE RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Training Random Forest...")

random_forest.fit(X_train, y_train)

print("Random Forest training completed.")

# Predictions
rf_pred = random_forest.predict(X_test)

# Prediction probabilities
rf_prob = random_forest.predict_proba(X_test)[:, 1]

print("Random Forest predictions completed.")

Training Random Forest...
Random Forest training completed.
Random Forest predictions completed.


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_prob)

print("Leakage-Safe Random Forest Performance")
print("---------------------------------------")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Leakage-Safe Random Forest Performance
---------------------------------------
Accuracy : 0.7130
Precision: 0.7835
Recall   : 0.6587
F1 Score : 0.7157
ROC-AUC  : 0.7859

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.78      0.71     16308
           1       0.78      0.66      0.72     19796

    accuracy                           0.71     36104
   macro avg       0.72      0.72      0.71     36104
weighted avg       0.72      0.71      0.71     36104



In [7]:
# ============================================
# 5. FEATURE IMPORTANCE
# ============================================

# Get encoded feature names
feature_names = preprocessor.get_feature_names_out()

# Get importance values
importance_values = random_forest.feature_importances_

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

# Sort by importance
feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

# Display top 15 features
print("Top 15 Risk Drivers")
print("--------------------")

print(
    feature_importance.head(15).to_string(index=False)
)

Top 15 Risk Drivers
--------------------
                                  Feature  Importance
       remainder__Order Item Profit Ratio    0.101486
           remainder__Order Item Discount    0.086506
      remainder__Order Item Discount Rate    0.076935
 remainder__Days for shipment (scheduled)    0.073626
categorical__Shipping Mode_Standard Class    0.067667
               remainder__Order_DayOfWeek    0.067007
                   remainder__Order_Month    0.060847
   categorical__Shipping Mode_First Class    0.048450
           remainder__Order Item Quantity    0.030662
                 remainder__Product Price    0.021764
      remainder__Order Item Product Price    0.021584
  categorical__Shipping Mode_Second Class    0.021073
                    remainder__Order_Year    0.018648
   categorical__Customer Segment_Consumer    0.015910
  categorical__Customer Segment_Corporate    0.014185


In [14]:
# ============================================
# TOP RISK DRIVERS - LEAKAGE-SAFE MODEL
# ============================================

feature_names = preprocessor.get_feature_names_out()

importance_values = random_forest.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
}).sort_values(
    "Importance",
    ascending=False
)

print("Top 15 Leakage-Safe Risk Drivers")
print("--------------------------------")

print(
    feature_importance.head(15).to_string(index=False)
)

ValueError: All arrays must be of the same length

In [15]:
# ============================================
# RETRAIN LEAKAGE-SAFE RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Retraining leakage-safe Random Forest...")

random_forest.fit(X_train, y_train)

rf_pred = random_forest.predict(X_test)
rf_prob = random_forest.predict_proba(X_test)[:, 1]

print("Leakage-safe Random Forest retrained successfully.")

Retraining leakage-safe Random Forest...
Leakage-safe Random Forest retrained successfully.


In [16]:
# ============================================
# TOP 15 LEAKAGE-SAFE RISK DRIVERS
# ============================================

feature_names = preprocessor.get_feature_names_out()
importance_values = random_forest.feature_importances_

print("Feature count:", len(feature_names))
print("Importance count:", len(importance_values))

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
}).sort_values(
    "Importance",
    ascending=False
)

print("\nTop 15 Leakage-Safe Risk Drivers")
print("--------------------------------")

print(
    feature_importance.head(15).to_string(index=False)
)

Feature count: 274
Importance count: 273


ValueError: All arrays must be of the same length

In [17]:
print("Random Forest features:", len(random_forest.feature_importances_))
print("Preprocessor feature names:", len(preprocessor.get_feature_names_out()))
print("X_train columns:", X_train.shape[1])

Random Forest features: 273
Preprocessor feature names: 274
X_train columns: 273


In [18]:
# Reload the corrected leakage-safe preprocessor

import joblib

preprocessor = joblib.load("../data/ml_preprocessor.pkl")

print("Corrected preprocessor loaded.")
print("Preprocessor features:", len(preprocessor.get_feature_names_out()))

Corrected preprocessor loaded.
Preprocessor features: 273


In [19]:
# ============================================
# TOP 15 LEAKAGE-SAFE RISK DRIVERS
# ============================================

feature_names = preprocessor.get_feature_names_out()
importance_values = random_forest.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
}).sort_values(
    "Importance",
    ascending=False
)

print("Top 15 Leakage-Safe Risk Drivers")
print("--------------------------------")

print(
    feature_importance.head(15).to_string(index=False)
)

Top 15 Leakage-Safe Risk Drivers
--------------------------------
                                  Feature  Importance
           remainder__Order Item Discount    0.107291
      remainder__Order Item Discount Rate    0.096324
               remainder__Order_DayOfWeek    0.087031
                   remainder__Order_Month    0.077931
 remainder__Days for shipment (scheduled)    0.071412
categorical__Shipping Mode_Standard Class    0.060136
   categorical__Shipping Mode_First Class    0.057150
           remainder__Order Item Quantity    0.036767
  categorical__Shipping Mode_Second Class    0.022642
                 remainder__Product Price    0.021915
      remainder__Order Item Product Price    0.021825
                    remainder__Order_Year    0.021708
   categorical__Customer Segment_Consumer    0.017643
  categorical__Customer Segment_Corporate    0.015655
categorical__Customer Segment_Home Office    0.013227


In [20]:
# ============================================
# FINAL LEAKAGE-SAFE RANDOM FOREST EVALUATION
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_prob)

print("FINAL LEAKAGE-SAFE RANDOM FOREST")
print("----------------------------------")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

FINAL LEAKAGE-SAFE RANDOM FOREST
----------------------------------
Accuracy : 0.7130
Precision: 0.7835
Recall   : 0.6587
F1 Score : 0.7157
ROC-AUC  : 0.7859

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.78      0.71     16308
           1       0.78      0.66      0.72     19796

    accuracy                           0.71     36104
   macro avg       0.72      0.72      0.71     36104
weighted avg       0.72      0.71      0.71     36104



In [ ]:
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ==========================================
# 1. Load ML Dataset
# ==========================================

df = pd.read_csv(
    "data/SupplyChain_ML_Features.csv"
)

print("ML dataset shape:", df.shape)


# ==========================================
# 2. Separate Features and Target
# ==========================================

target = "Late_delivery_risk"

X = df.drop(columns=[target])
y = df[target]


# ==========================================
# 3. Identify Feature Types
# ==========================================

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("\nCategorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))


# ==========================================
# 4. Create Preprocessor
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)


# ==========================================
# 5. Encode Features
# ==========================================

X_encoded = preprocessor.fit_transform(X)

print("Encoded feature shape:", X_encoded.shape)


# ==========================================
# 6. Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


# ==========================================
# 7. Create Random Forest Model
# ==========================================

random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)


# ==========================================
# 8. Train Model
# ==========================================

print("\nTraining Random Forest model...")

random_forest.fit(
    X_train,
    y_train
)


# ==========================================
# 9. Make Predictions
# ==========================================

y_pred = random_forest.predict(X_test)

y_prob = random_forest.predict_proba(X_test)[:, 1]


# ==========================================
# 10. Evaluate Model
# ==========================================

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)


# ==========================================
# 11. Display Model Performance
# ==========================================

print("\nRandom Forest Model Performance")
print("--------------------------------")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


# ==========================================
# 12. Save Model
# ==========================================

joblib.dump(
    random_forest,
    "data/random_forest_risk_model.pkl"
)

joblib.dump(
    preprocessor,
    "data/ml_preprocessor.pkl"
)


# ==========================================
# 13. Final Confirmation
# ==========================================

print("\nModel and preprocessor saved successfully.")

print("Model:")
print("data/random_forest_risk_model.pkl")

print("\nPreprocessor:")
print("data/ml_preprocessor.pkl")

Final Random Forest model saved successfully.
Risk driver file saved successfully.


In [22]:
import pandas as pd

df_check = pd.read_csv(
    "../data/DataCoSupplyChain_Cleaned.csv"
)

print("Rows:", len(df_check))
print("Columns:", len(df_check.columns))

Rows: 180519
Columns: 45
